# 🧠 Research-Quality 4-Class Brain MRI Tumor Classification

## 1. Problem Statement & Objective
Brain tumors are among the most aggressive and life-threatening oncological conditions. Accurate automatic classification of magnetic resonance imaging (MRI) scans into specific histological tumor categories—**Glioma**, **Meningioma**, **Pituitary**, or **No Tumor**—is essential for assisting neuro-radiological diagnostic workflows.

This project implements a multi-model comparative deep learning benchmark in TensorFlow/Keras evaluating:
1. **Model 1 — Custom Baseline CNN** (built from scratch)
2. **Model 2 — ResNet50** (ImageNet pretrained, 2-Stage Transfer Learning)
3. **Model 3 — DenseNet121** (ImageNet pretrained, 2-Stage Transfer Learning)
4. **Model 4 — EfficientNetV2B0** (ImageNet pretrained, 2-Stage Transfer Learning)


In [ ]:
import os, sys
sys.path.append("..")
import matplotlib.pyplot as plt
from PIL import Image
from src import config
from src.utils import set_seeds, get_hardware_info
from src.dataset_analysis import analyze_dataset

set_seeds(config.SEED)
get_hardware_info()

## 2. Dataset Quality & Quality Control Inspection
We analyze the recursive image structure, format consistency, resolution distribution, exact MD5 duplicates, and class imbalance.

In [ ]:
dataset_stats = analyze_dataset(data_dir=config.DATA_DIR)
# Display saved dataset analysis plot
img_analysis = Image.open(os.path.join(config.PLOTS_DIR, "dataset_analysis.png"))
plt.figure(figsize=(12, 5))
plt.imshow(img_analysis)
plt.axis("off")
plt.show()

## 3. Representative Brain MRI Samples per Class

In [ ]:
img_samples = Image.open(os.path.join(config.PLOTS_DIR, "dataset_samples.png"))
plt.figure(figsize=(12, 10))
plt.imshow(img_samples)
plt.axis("off")
plt.show()

## 4. Stratified Data Split & tf.data Pipeline
We enforce a **70% Train / 15% Val / 15% Test** stratified split without data leakage.

In [ ]:
from src.data_loader import load_dataset_file_paths, get_class_weights, create_tf_dataset
from src.preprocessing import get_preprocessing_function
from src.augmentation import build_augmentation_pipeline

splits = load_dataset_file_paths()
class_weights = get_class_weights(splits["train_labels"])
aug_pipeline = build_augmentation_pipeline(enabled=True)

## 5. Model Building & Experimental Training
We instantiate and train all 4 architectures.

In [ ]:
from src.models import build_baseline_cnn, build_transfer_learning_model
from src.train import train_model

print("Baseline CNN Architecture:")
baseline_model = build_baseline_cnn()
baseline_model.summary()

## 6. Model Comparison & Metrics Table

In [ ]:
import pandas as pd
comp_csv = os.path.join(config.RESULTS_DIR, "model_comparison.csv")
if os.path.exists(comp_csv):
    df_res = pd.read_csv(comp_csv)
    display(df_res)

## 7. Qualitative Error Analysis

In [ ]:
err_plot = os.path.join(config.PLOTS_DIR, "error_analysis_samples.png")
if os.path.exists(err_plot):
    img_err = Image.open(err_plot)
    plt.figure(figsize=(10, 8))
    plt.imshow(img_err)
    plt.axis("off")
    plt.show()

## 8. Model Interpretability via Grad-CAM

In [ ]:
gradcam_plot = os.path.join(config.GRADCAM_DIR, "EfficientNetV2B0_gradcam_grid.png")
if not os.path.exists(gradcam_plot):
    # fallback to available gradcam plot
    gradcam_files = [f for f in os.listdir(config.GRADCAM_DIR) if f.endswith(".png")]
    if gradcam_files:
        gradcam_plot = os.path.join(config.GRADCAM_DIR, gradcam_files[0])

if os.path.exists(gradcam_plot):
    img_gc = Image.open(gradcam_plot)
    plt.figure(figsize=(12, 10))
    plt.imshow(img_gc)
    plt.axis("off")
    plt.show()

## 9. Conclusion & Research Disclaimers
1. **Transfer Learning Superiority**: Pretrained feature extractors (ResNet50, DenseNet121, EfficientNetV2B0) significantly outperform random initialization baselines.
2. **Interpretability Validation**: Grad-CAM visual overlays confirm that model attention is focused on intracranial tissue and tumor margins.
3. **Research Disclaimer**: This system is developed for educational and research exploration and is NOT approved for direct clinical diagnostic deployment.